# RAG permutation study - main run

Four cells. Run **Setup**, check **Status**, set `SESSION`, run **Go**, then
**Save** before you lose the runtime.

The main grid is ~38,500 generations plus the oracle's scoring - far more than a
free-tier session. It is split into four sittings, none over ~4.5h.

| session | arms | ~time | why grouped this way |
|---|---|---|---|
| **1** | full, rerank_topk, random_drop, placebo_pos x3 | ~4h | selections are instant or near-instant, so nearly all the time goes into generation |
| **2** | provence_rerank, provence_full, llmlingua2 | ~3h | these three load their own encoders; grouping them pays that cost once |
| **3** | llm_pruner, loo_oracle | ~4.5h | both drive the generator during selection; the oracle alone is ~3h of scoring |
| **4** | everything | ~10 min | all cached by now - just rebuilds the grid and writes the CSV |

**Why grouping matters:** only *generations* are cached. Provence and LLMLingua-2
re-run their encoders from scratch every session, so bundling them with cheap arms
would pay ~45 min of selection again on every resume.

Nothing is lost when a session dies. The cache is snapshotted to Drive every two
minutes and re-running a session picks up exactly where it stopped.


## Setup

Run once per session. GPU check, Drive, clone, dependencies, cache restore,
autosync. Takes ~4 min, most of it model weights.


In [ ]:
import os, shutil, sqlite3, subprocess, threading

# ---- GPU -------------------------------------------------------------
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4.'
print(torch.cuda.get_device_properties(0).name,
      f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ---- Drive -----------------------------------------------------------
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/rag-permutation-study'
os.makedirs(DRIVE_DIR, exist_ok=True)

REPO, WORK = 'Rahulrayy/rag-permutation-study', '/content/rag'
LOCAL_CACHE = f'{WORK}/cache/generations.sqlite'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'generations.sqlite')


def sh(cmd, redact=None):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if redact:
        out = out.replace(redact, '***')
    if out:
        print(out)
    if r.returncode:
        raise RuntimeError(f'{cmd[0]} failed ({r.returncode})')


# ---- code ------------------------------------------------------------
token = userdata.get('GH_TOKEN')
AUTH = f'https://{token}@github.com/{REPO}.git'
if not os.path.exists(WORK):
    sh(['git', 'clone', AUTH, WORK], redact=token)
    sh(['git', '-C', WORK, 'remote', 'set-url', 'origin',
        f'https://github.com/{REPO}.git'])
else:
    sh(['git', '-C', WORK, 'pull', AUTH, 'master'], redact=token)
del token, AUTH
os.chdir(WORK)

# ---- deps ------------------------------------------------------------
# torch is skipped: Colab's build matches its own CUDA runtime.
os.system("grep -v '^torch' requirements.txt > /tmp/req.txt")
os.system('pip install -q -r /tmp/req.txt')
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)   # Provence refuses to load without it

import transformers
assert int(transformers.__version__.split('.')[0]) >= 5, (
    'transformers 5.x required: generate.py uses the `dtype=` kwarg')


def _count(path):
    if not os.path.exists(path):
        return -1
    try:
        con = sqlite3.connect(path)
        n = con.execute('SELECT COUNT(*) FROM generations').fetchone()[0]
        con.close()
        return n
    except sqlite3.Error:
        return -1


def sync_cache(verbose=True):
    """Snapshot the live cache to Drive via SQLite's online backup API.

    Not a file copy: a copy of the .sqlite silently drops whatever is still in
    the -wal, which is routinely the larger of the two. The backup API is built
    to snapshot a database that is being written to, so this is safe mid-run.
    """
    if not os.path.exists(LOCAL_CACHE):
        return 0
    tmp = '/content/_snap.sqlite'
    s, d = sqlite3.connect(LOCAL_CACHE), sqlite3.connect(tmp)
    with d:
        s.backup(d)
    n = d.execute('SELECT COUNT(*) FROM generations').fetchone()[0]
    d.close(); s.close()
    shutil.copy(tmp, DRIVE_CACHE)
    os.remove(tmp)
    if verbose:
        print(f'[sync] {n} generations -> Drive')
    return n


# ---- cache restore, whichever copy is ahead --------------------------
# Guard against re-running this cell mid-session: Drive may be BEHIND local,
# and a blind restore would throw away everything generated since the last
# snapshot. Whichever side has more rows wins.
os.makedirs(os.path.dirname(LOCAL_CACHE), exist_ok=True)
local_n, drive_n = _count(LOCAL_CACHE), _count(DRIVE_CACHE)
if drive_n > local_n:
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    print(f'restored {drive_n} generations from Drive')
elif local_n > drive_n:
    print(f'local cache is ahead ({local_n} vs {max(drive_n, 0)}) - keeping it')
    sync_cache()
else:
    print(f'cache in sync ({max(local_n, 0)} generations)')

# ---- autosync --------------------------------------------------------
_stop = threading.Event()


def _loop(every_s):
    while not _stop.wait(every_s):
        try:
            sync_cache(verbose=False)
        except Exception as exc:
            print('[sync] failed, continuing:', exc)


if not any(t.name == 'autosync' for t in threading.enumerate()):
    threading.Thread(target=_loop, args=(120,), name='autosync',
                     daemon=True).start()

print('\nSetup complete. Autosync every 2 min.')


## Status

Where the run is. Safe to re-run at any time - it only reads.


In [ ]:
MAIN_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
# Grid estimate: 300 queries -> ~267 after the memorization filter, 11 arms
# after placebo_pos expands to three, 3 budgets, 5 permutations, less `full`'s
# dedup across budgets. Approximate on purpose - the exact figure depends on
# how many queries the filter removes.
TARGET = 38500

con = sqlite3.connect(LOCAL_CACHE)
rows = dict(con.execute('SELECT model, COUNT(*) FROM generations GROUP BY model'))
con.close()

done = rows.get(MAIN_MODEL, 0)
pct = 100 * done / TARGET
bar = '#' * int(pct / 2.5) + '.' * (40 - int(pct / 2.5))

print('cached generations by model')
for m, n in sorted(rows.items(), key=lambda kv: -kv[1]):
    mark = '  <- the main run' if m == MAIN_MODEL else ''
    print(f'   {m:28} {n:6}{mark}')

print(f'\nmain run  [{bar}] {done:,} / ~{TARGET:,}  ({pct:.0f}%)')
print('\n(Estimate. Session 4 is what actually decides the grid is complete,')
print(' by rebuilding it and finding every prompt already cached.)')


## Go

Set `SESSION` and run. Re-running a session is always safe: cached prompts
replay for free and it continues from where it stopped.

No `generations.csv` appears until session 4 - the grid is incomplete before
then by design, and a partial CSV would be worse than none. The work lives in
the cache.


In [ ]:
SESSION = 1   # <--- 1, 2, 3 or 4

SESSIONS = {
    1: ('cheap-selection arms  (~4h)',
        ['nocontext', 'full', 'rerank_topk', 'random_drop', 'placebo_pos']),
    2: ('encoder arms  (~3h)',
        ['nocontext', 'provence_rerank', 'provence_full', 'llmlingua2']),
    3: ('generator-dependent arms  (~4.5h)',
        ['nocontext', 'llm_pruner', 'loo_oracle']),
    4: ('assemble everything, writes the CSV  (~10 min)', None),
}

label, arms = SESSIONS[SESSION]
print(f'session {SESSION}: {label}\n')

cmd = ['python', '-m', 'src.run', '--config', 'configs/main_colab.yaml']
if arms:
    cmd += ['--arms'] + arms
print(' '.join(cmd), '\n')

# Streamed, not captured, so progress is visible while it runs.
os.system(' '.join(cmd))
sync_cache()


## Save

**Run this before you close the tab or lose the runtime.** The autosync thread
only lives as long as the kernel.


In [ ]:
sync_cache()

if os.path.exists(f'{WORK}/results'):
    shutil.copytree(f'{WORK}/results', os.path.join(DRIVE_DIR, 'results'),
                    dirs_exist_ok=True)
    print('results -> Drive')


## After session 4

Once the CSV exists, this reproduces the week-1 gate on the main run. Compare
against `results/pilot_w1/gate_report.txt` - the pilot's median within-query SD
was **0.0263** on n=100, `full` arm, unfiltered.

Expect the main run's figure to be **higher**, and not because the effect grew:
the main run applies the memorization filter, which removes queries answerable
from parametric memory - exactly the queries most likely to be stable under
permutation. That inflation is a property of the population, not the finding.
ANALYSIS_PLAN section 9 records this.


In [ ]:
os.system('python -m src.gate results/main_hotpotqa/generations.csv --budget 3')
